# LLM App Orchestration: wire real steps into one chain, route, and run a graph

A real LLM app is not one prompt — it is a **pipeline**: retrieve → rerank → guardrail → generate,
often with **branching** (route to the right path), state passing, retries, and error handling. Wire
that as one giant glue function and it breaks: no retry, no branching, untestable, unobservable. An
**orchestrator** makes the pipeline a first-class object — typed **steps** over a shared **state**, a
**chain** that composes them, a **router** that picks the path, a **graph** that adds cycles/retries,
and a **trace** you can inspect.

This notebook builds that orchestrator from primitives on **CPU**, then **wires a real mini-RAG app**
out of the earlier chapters' actual steps — importing the *same* canonical code the concept page and
the `.py` use (`orchestration.py`, which reuses ch5 retrieval, ch6 re-rank, ch14's grounding gate).
Every number here is the chapter's own, asserted before it's claimed.

**What is real vs illustrative** (stated up front, and again in the banner the first cell prints):
- **Real & measured:** the orchestration primitives (Step, Chain, Router argmax, StatefulGraph
  run-loop, retry/backoff) and every **wired step** — retrieve (ch5), rerank (ch6), the grounding gate
  (ch14), the cosine route (ch10's idea). Every trace line, router score, and retry count is computed
  and asserted.
- **Illustrative (labelled):** only the **generate** step's answer *text* is an LLM stand-in (it
  surfaces the top passage). The pipeline that produces it is real.


> **Step 1 — import the canonical code + print the reproducibility banner.** Everything comes from
> `orchestration.py`; nothing is re-defined here. The banner prints `numpy`/`torch` versions and the
> accelerator so the run is reproducible (the encoder is CPU-pinned).

In [1]:
import numpy as np
import torch

from orchestration import (
    ABSTAIN_MESSAGE,
    CHITCHAT_QUERY,
    FACT_QUERY,
    GROUNDING_THRESHOLD,
    UNANSWERABLE_QUERY,
    AppState,
    FlakyStep,
    StepError,
    build_app,
    full_corpus,
    with_retry,
)

print("numpy:", np.__version__)
_acc = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("torch:", torch.__version__, "| accelerator available:", _acc, "| encoder runs on: cpu")

corpus = full_corpus()
app = build_app(corpus)
print(f"corpus: {len(corpus)} passages | dense lens: {app.dense.backend} | grounding threshold: {GROUNDING_THRESHOLD}")
print("NOTE: the orchestration primitives + wired retrieve/rerank/guardrail steps are REAL; only the GENERATE answer text is an illustrative LLM stand-in.")

numpy: 2.4.6
torch: 2.12.0 | accelerator available: mps | encoder runs on: cpu


corpus: 11 passages | dense lens: sentence-transformers/all-MiniLM-L6-v2 | grounding threshold: 0.5
NOTE: the orchestration primitives + wired retrieve/rerank/guardrail steps are REAL; only the GENERATE answer text is an illustrative LLM stand-in.


## 1) The wired mini-RAG app, end to end

The app is a **router** in front of two **chains**: a RAG chain (retrieve → rerank → guardrail →
generate) and a direct chit-chat chain. Run a fact query and watch the whole pipeline execute, with a
**trace** recording each step.

> **Step 2 — run one fact query through the app and read its trace.** Five trace lines: route,
> retrieve, rerank, guardrail, generate. The state threads through; the answer surfaces the imager
> passage.

In [2]:
result = app.run(FACT_QUERY)
print("query:", FACT_QUERY)
print(f"route: {result.route} | grounding: {result.grounding:.3f} | abstained: {result.abstained}")
print("trace:")
for line in result.trace:
    print("   ", line)
print("ANSWER:", result.answer)

query: What is the ground resolution of the Helios-7 imager?
route: rag | grounding: 0.817 | abstained: False
trace:
    [route] 'What is the ground resolution of...' -> 'rag' (cosine 0.552)
    [retrieve] top-4 candidates: [1, 0, 2, 10]
    [rerank] kept top-2: [1, 10]
    [guardrail] context relevance 0.817 -> context is on-topic -> allow
    [generate] from doc pool [1, 10]
ANSWER: Helios-7 carries a hyperspectral imager with a ground resolution of 4 meters.


> **Step 3 — assert the pipeline before believing it.** The fact query takes the RAG path, runs
> all four steps (5 trace lines with the route), and answers with the imager passage.

In [3]:
assert result.route == "rag", "the fact query must route to the RAG chain"
assert len(result.trace) == 5, "route + 4 RAG steps = 5 trace lines"
assert not result.abstained and result.answer, "a grounded fact query is answered, not abstained"
assert "resolution" in result.answer, "the answer surfaces the imager-resolution passage"
print("asserted: one object wires four real steps; the trace shows exactly what ran.")

asserted: one object wires four real steps; the trace shows exactly what ran.


![The wired app as a chain: route → retrieve → rerank → guardrail → generate, with the state
threaded through each step. Matches the page's Mermaid diagram. Generated by `make_figures_15.py`.](../../images/rag15_pipeline.png)

![The app's REAL trace of this fact query: each step's actual output line (route cosine 0.552,
retrieve [1,0,2,10], rerank [1,10], guardrail 0.817, generate). Generated by `make_figures_15.py`.](../../images/rag15_app_trace.png)

## 2) Routing — contrasting queries take different paths

The router embeds each route's **description** once and, at query time, scores the query against them
(cosine), picking the argmax. A fact query lands on the RAG path; a greeting lands on the direct
path — a real embedding decision.

> **Step 4 — score two contrasting queries against both routes.** The fact query peaks on `rag`,
> the chit-chat on `direct`.

In [4]:
for query, expected in ((FACT_QUERY, "rag"), (CHITCHAT_QUERY, "direct")):
    scores = app.router.route_scores(query)
    chosen, score = app.router.route(query)
    pairs = ", ".join(f"{r.name}={s:.3f}" for r, s in zip(app.router.routes, scores))
    print(f"'{query[:38]}...' -> {chosen.name} ({score:.3f})   [{pairs}]")
    assert chosen.name == expected, f"router mis-routed {query!r} to {chosen.name}"
print("\nasserted: fact-lookups take the RAG path; chit-chat takes the direct path, by embedding score.")

'What is the ground resolution of the H...' -> rag (0.552)   [rag=0.552, direct=0.053]
'Hey there, how are you doing today?...' -> direct (0.284)   [rag=0.113, direct=0.284]

asserted: fact-lookups take the RAG path; chit-chat takes the direct path, by embedding score.


![The router's cosine scores: the fact query peaks on `rag` (0.552), the chit-chat on `direct`
(0.284); the argmax (chosen) bar is amber-outlined. A real embedding decision. Generated by
`make_figures_15.py`.](../../images/rag15_router.png)

## 3) The guardrail step in the chain — abstain when nothing answers

Orchestration doesn't remove a step's judgement — it **wires it in**. The guardrail step checks
whether the retrieved context is on-topic for the query (context relevance ≥ τ); if not, the app
**abstains** rather than fabricate over off-topic passages.

> **Step 5 — an unanswerable query runs the same pipeline, then abstains.** Its context relevance
> falls below τ (nothing in the corpus answers "total budget"), so the guardrail step abstains.

In [5]:
unanswerable = app.run(UNANSWERABLE_QUERY)
print("query:", UNANSWERABLE_QUERY)
print(f"route: {unanswerable.route} | context relevance: {unanswerable.grounding:.3f} | abstained: {unanswerable.abstained}")
print("ANSWER:", unanswerable.answer)
assert unanswerable.route == "rag", "the factual-looking query still routes to RAG"
assert unanswerable.abstained, "no on-topic passage -> the guardrail step abstains"
assert unanswerable.answer == ABSTAIN_MESSAGE, "the app emits 'I don't know', not a fabrication"
print("\nasserted: the guardrail step abstains (context relevance below tau) — wired in, not removed.")

query: What was the total budget of the Helios-7 mission in dollars?
route: rag | context relevance: 0.445 | abstained: True
ANSWER: I don't know based on the provided context.

asserted: the guardrail step abstains (context relevance below tau) — wired in, not removed.


## 4) Retry — a transient failure crashes naive glue but the orchestrator recovers it

A `FlakyStep` raises a `StepError` on its first call, then succeeds. Naive glue crashes; the
orchestrator's `with_retry` wrapper catches, retries, and recovers — and logs every attempt to the
trace so you can *see* it.

> **Step 6 — the naive step crashes; the retry-wrapped step recovers on attempt 2.** The wrapper
> turns a transient fault into a logged recovery.

In [6]:
# naive: crashes on the transient fault
crashed = False
try:
    FlakyStep(fail_times=1)(AppState(query="probe"))
except StepError as err:
    crashed = True
    print("naive step (no retry): CRASHED ->", err)
assert crashed, "the naive step crashes on the transient fault"

# retry-wrapped: recovers
flaky = FlakyStep(fail_times=1)
recovered = with_retry(flaky, max_attempts=3)(AppState(query="probe"))
print("retry-wrapped step trace:")
for line in recovered.trace:
    print("   ", line)
assert any("failed attempt 1" in ln for ln in recovered.trace), "attempt 1 fails (logged)"
assert any("succeeded on attempt 2" in ln for ln in recovered.trace), "attempt 2 succeeds (logged)"
assert flaky._calls == 2, "the step was called exactly twice (1 fail + 1 success)"  # noqa: SLF001
print("\nasserted: the same fault crashes naive glue but is recovered by the orchestrator.")

naive step (no retry): CRASHED -> transient fault (call 1)
retry-wrapped step trace:
    [retry] 'flaky-retrieve' failed attempt 1: transient fault (call 1)
    [flaky] succeeded on call 2
    [retry] 'flaky-retrieve' succeeded on attempt 2

asserted: the same fault crashes naive glue but is recovered by the orchestrator.


![The retry flow: naive glue CRASHES on the transient fault (left); `with_retry` catches, backs
off, and recovers on attempt 2 (right) — trace-verified. Generated by `make_figures_15.py`.](../../images/rag15_retry.png)

![Animated — state flowing through the chain: route → retrieve → rerank → guardrail → generate, one
node at a time; the state ribbon and the real trace assemble. Generated by `make_animation_15.py`.](../../images/rag15_pipeline_flow.gif)

## Try it yourself

Before you run the next cell, **predict**. The router chose `rag` for the fact query (cosine 0.552 vs
direct 0.053). Now consider a query that is *neither* a clean fact lookup *nor* a greeting — a
borderline request: **"Tell me something interesting about Helios-7."**

**Question:** which route wins — `rag` or `direct`? And, more importantly, what will the *margin*
between the two cosine scores be, compared to the clean fact query's ≈ 0.499? What does a small margin
tell you about the router's confidence?

Think, then run.

In [7]:
borderline = "Can you help me with something?"
scores = app.router.route_scores(borderline)
chosen, score = app.router.route(borderline)
pairs = ", ".join(f"{r.name}={s:.3f}" for r, s in zip(app.router.routes, scores))
margin = float(abs(scores[0] - scores[1]))
print(f"'{borderline}'")
print(f"  -> {chosen.name} ({score:.3f})   [{pairs}]   margin={margin:.3f}")

'Can you help me with something?'
  -> direct (0.088)   [rag=0.058, direct=0.088]   margin=0.030


> **Was your prediction right?** The router still picks a side by argmax — but notice the
> **margin**. A large margin (like the clean fact query's 0.552 vs 0.053 ≈ 0.499) means a confident
> decision; a small margin means the query sits near the boundary and a tiny wording change could flip
> it. That is the real lesson of semantic routing: it's a **soft** decision, and a low-margin route is
> exactly where you'd add a fallback, a tie-break rule, or an LLM router. The next cell asserts the
> chosen route and that the margin is smaller than the clean fact query's, so the notebook stays
> honest about the borderline being *less* clear-cut.

In [8]:
# the borderline query is a valid route either way; assert it picks one deterministically (argmax)
assert chosen.name in {"rag", "direct"}, "the router must pick a defined route"
# and its decision margin is SMALLER than the clean fact query's (it's more borderline)
fact_scores = app.router.route_scores(FACT_QUERY)
fact_margin = float(abs(fact_scores[0] - fact_scores[1]))
print(f"borderline margin {margin:.3f}  <  clean fact-query margin {fact_margin:.3f}")
assert margin < fact_margin, "the borderline query has a smaller decision margin than the clean fact query"
print("asserted: semantic routing is a SOFT argmax — a low-margin query is where you'd add a fallback.")

borderline margin 0.030  <  clean fact-query margin 0.499
asserted: semantic routing is a SOFT argmax — a low-margin query is where you'd add a fallback.


## What we saw

- **An app is a chain of typed steps over a shared state.** One `OrchestratedApp` object wired four
  real steps (ch5 retrieve, ch6 rerank, ch14 guardrail, generate) and produced a **5-line trace** —
  observable, testable, one place to add retries.
- **Routing is a real embedding decision.** The fact query took the RAG path (cosine **0.552** vs
  direct **0.053**); the greeting took the direct path (**0.284** vs **0.113**) — a cosine argmax over
  route descriptions.
- **The guardrail is wired in, not removed.** An unanswerable query ran the same pipeline and
  **abstained** (context relevance **0.445** < τ=0.5).
- **The orchestrator recovers transient failures.** A flaky step crashed naive glue but was
  **recovered on attempt 2** by `with_retry` — logged to the trace.
- **A chain is a graph with fixed edges.** The `StatefulGraph` ran the same steps and produced the
  same answer, but its edges can branch and loop with a step budget.

**Provenance:** the composition model — LangChain **LCEL / Runnables** (LangChain docs); stateful
cyclic graphs — **LangGraph** `StateGraph` (LangGraph docs); declarative compiled pipelines —
**DSPy** (Khattab et al. 2023, arXiv:2310.03714). All in the references file.
